In [0]:
import pyspark.sql.functions as F

dbutils.widgets.text("bronze_catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

dbutils.widgets.text("silver_catalog", "dbr_dev")
dbutils.widgets.text("silver_schema", "artemzharkov10_silver")


BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")


BRONZE_TABLE = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_history_weather_demo"
SILVER_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_history_weather_demo"

df_bronze = spark.read.table(BRONZE_TABLE)

df_silver = df_bronze.filter(
    F.col("grid_id").isNotNull() & 
    F.col("weather_time").isNotNull() & 
    F.col("soil_temperature_c").isNotNull() &
    (F.col("precipitation_mm") >= 0)
)

df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
print(f"Silver таблица {SILVER_TABLE} создана. Валидных записей: {df_silver.count()}")